# Semana 12: análise do braço IsoMax+ (E10, CPU)

Roda `analysis/exploratoria/e10_isomax.py`, que segue `prereg/plano_isomax.md`. **É exploratório**: não aplica categorias do pré-registro. Compara o R1 treinado com a IsoMax+ (sementes 1 a 3) com o R1 confirmatório das mesmas sementes: τ, cobertura, conjunto vazio, taxa de erro, mIoU, erro não sinalizado e AUROC_e da MSP; para a IsoMax+, também o AUROC_e do escore de distância mínima. As diferenças têm intervalos de 90% e 95% por reamostragem de unidades dentro dos eventos, com os mesmos sorteios para os dois braços. Antes de gravar, o script confere τ e o AUROC_e da validação do R1 contra a rodada confirmatória.

Precisa da saída de `semana12_isomax.ipynb` (pasta `resultados/semana12/isomax`, com o manifesto).

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive e confere RAM, disco e pastas |
| 2 | Clona o repositório, confere os scripts congelados e o plano, e roda os testes do E10 |
| 3 | Copia as margens da semana 6 e as da IsoMax+ para o disco local, conferidas pelos manifestos |
| 4 | Roda a análise (cerca de 15 minutos) |
| 5 | Copia as saídas e o log para `resultados/semana12/isomax_analise` |

Use um ambiente com pelo menos 12 GB de RAM; GPU não é necessária. Se alguma célula der erro, pare e guarde a saída inteira.

In [ ]:
# Célula 1 de 5: Drive, RAM, disco e pastas
import os, re, json, time, hashlib, shutil, subprocess, sys, csv
import psutil
from importlib.metadata import version

from google.colab import drive
drive.mount('/content/drive')
RAIZ = '/content/drive/MyDrive/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

MARGENS_DRIVE = f'{RAIZ}/resultados/semana6/margens'
CHAVE = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
CONFIRMATORIA = f'{RAIZ}/resultados/semana6/analise'
ISO_DRIVE = f'{RAIZ}/resultados/semana12/isomax'
SAIDA_DRIVE = f'{RAIZ}/resultados/semana12/isomax_analise'
for caminho in (CHAVE, f'{CONFIRMATORIA}/resultados.json', f'{MARGENS_DRIVE}/manifesto_sha256.csv', f'{ISO_DRIVE}/manifesto_sha256.csv'):
    assert os.path.isfile(caminho), f'Arquivo não encontrado: {caminho}'
assert not (os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE)), (
    'A pasta resultados/semana12/isomax_analise já tem arquivos. Não sobrescrevo: me avise antes de rodar de novo.')

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

ram = psutil.virtual_memory().total / 1e9
disco = shutil.disk_usage('/content')[2] / 1e9
print(f'RAM da sessão: {ram:.1f} GB. Disco local livre: {disco:.0f} GB.')
assert ram >= 12, 'RAM abaixo de 12 GB. Use um ambiente com mais memória.'
assert disco > 30, 'Disco local com menos de 30 GB livres.'

In [ ]:
# Célula 2 de 5: repositório, scripts congelados, plano e testes
REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()

tabela = open(f'{REPO}/analysis/README.md', encoding='utf-8').read()
registrados = dict(re.findall(r'\| `([a-z_]+\.py)` \| `([0-9a-f]{64})` \|', tabela))
assert set(registrados) == {'analise_congelada.py', 'nucleo.py', 'reamostragem.py', 'categorias.py'}, registrados.keys()
for nome, sha in registrados.items():
    assert sha256_arquivo(f'{REPO}/analysis/{nome}') == sha, f'{nome} é diferente do congelado. Não rode.'
data_plano = subprocess.run(['git', '-C', REPO, 'log', '--diff-filter=A', '-1', '--format=%cI %h', '--',
                             'prereg/plano_isomax.md'], capture_output=True, text=True, check=True).stdout.strip()
assert data_plano, 'prereg/plano_isomax.md não está no repositório.'
SHA_E10 = sha256_arquivo(f'{REPO}/analysis/exploratoria/e10_isomax.py')
r = subprocess.run([sys.executable, f'{REPO}/analysis/exploratoria/teste_e10.py'], capture_output=True, text=True)
print(r.stdout[-300:])
assert r.returncode == 0, 'Os testes do E10 falharam. Pare e guarde esta saída.'
print(f'Repositório no commit {commit[:7]}. Scripts congelados conferem. Plano adicionado em {data_plano}.')

In [ ]:
# Célula 3 de 5: margens no disco local, conferidas pelos manifestos
def copiar(pasta_drive, local, esperado=None, filtro=lambda n: True):
    os.makedirs(local, exist_ok=True)
    man = [l for l in csv.DictReader(open(f'{pasta_drive}/manifesto_sha256.csv', newline='')) if filtro(l['arquivo'])]
    if esperado is not None:
        assert len(man) == esperado, f'{pasta_drive}: {len(man)} arquivos no manifesto, esperado {esperado}.'
    for l in man:
        destino = f'{local}/{l["arquivo"]}'
        if not (os.path.exists(destino) and os.path.getsize(destino) == int(l['bytes'])):
            shutil.copy(f'{pasta_drive}/{l["arquivo"]}', destino + '.parcial')
            os.replace(destino + '.parcial', destino)
        assert sha256_arquivo(destino) == l['sha256'], f'{l["arquivo"]} diferente do manifesto.'
    return len(man)

t0 = time.time()
LOCAL = '/content/margens'
ISO = '/content/isomax'
n6 = copiar(MARGENS_DRIVE, LOCAL, esperado=64)
n_iso = copiar(ISO_DRIVE, ISO, esperado=12, filtro=lambda n: n.startswith('iso_s') and n.endswith('.npz'))
print(f'{n6} arquivos da semana 6 e {n_iso} da IsoMax+ copiados e conferidos em {(time.time() - t0) / 60:.1f} min.')

In [ ]:
# Célula 4 de 5: análise (cerca de 15 minutos)
SAIDA = '/content/semana12_analise'
LOG = '/content/semana12_e10.log'
if os.path.isdir(SAIDA):
    shutil.rmtree(SAIDA)   # saída local de uma tentativa interrompida
cmd = (f'cd {REPO} && set -o pipefail && python analysis/exploratoria/e10_isomax.py --repo . --margens {LOCAL} '
       f'--isomax {ISO} --chave "{CHAVE}" --confirmatoria "{CONFIRMATORIA}" --saida {SAIDA} 2>&1 | tee {LOG}')
get_ipython().system(f"bash -c '{cmd}'")
assert os.path.isfile(f'{SAIDA}/e10.json'), 'A análise não terminou. Guarde o log e me mande.'

In [ ]:
# Célula 5 de 5: cópia para o Drive
os.makedirs(SAIDA_DRIVE, exist_ok=True)
shutil.copytree(SAIDA, SAIDA_DRIVE, dirs_exist_ok=True)
shutil.copy(LOG, f'{SAIDA_DRIVE}/semana12_e10.log')
INFO = {'commit_repositorio': commit, 'plano': 'prereg/plano_isomax.md', 'plano_adicionado': data_plano,
        'e10_sha256': SHA_E10, 'congelados_sha256': registrados,
        'versoes': {q: version(q) for q in ('numpy', 'scipy')},
        'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rodada.json', 'w'), indent=2)
for raiz, _, arquivos in os.walk(SAIDA_DRIVE):
    for a in sorted(arquivos):
        print(os.path.relpath(os.path.join(raiz, a), SAIDA_DRIVE))
print(f'\nSaída copiada para {SAIDA_DRIVE}.')